# 4 · Learning from a single example

In notebook 3 we looked up **individual** past cases that resemble SO-1042. Sometimes, the question is simpler: "what *kind* of
problem is this, so who do I call first?" This matters if you're an analyst tasked with finding the cause of an issue.

That's a **classification** problem. Given a problem order's paths, predict its issue type:
`customs_delay`, `dispatch_hold` or `damaged_in_transit`. In this notebook we:

1. summarise all past cases into one hypervector per issue type, and use them to classify
   six new orders;
2. watch the classifier get two of them wrong, and see why;
3. add **one** newly resolved case, SO-1042, and watch both mistakes get fixed, with no
   retraining.

## The key terms we need to understand

| Term | What it means |
|---|---|
| **prototype** | A hypervector that summarises a whole class, category or label. This is done by bundling (⊕) all of the motif hypervectors of every past order (labelled data) in that class. To classify a new order, we compare it with each prototype and pick the most similar. |
| **one-shot learning** | Learning from a single example. For a prototype, that means bundling one more motif into the existing hypervector: one addition, with no training loop and no gradient descent. |

## Setup

We rebuild the graph and codebook and reuse the path encoder from notebook 3.
`HELD_OUT` contains the six orders we'll use to test the classifier.


In [1]:
from sales_orders import NODES, EDGES, HELD_OUT, write_graph, graph_config, make_codebook, path, edge_rows

import lancedb
import polars as pl
import torch
import torchhd
from lance_csr import CypherEngine

pl.Config.set_tbl_rows(20)  # print every row of our small tables
pl.Config.set_fmt_str_lengths(200)  # and the HAD_ISSUE notes in full

db = lancedb.connect("sales.lancedb")
write_graph(db)
engine = CypherEngine(graph_config(), {name: db.open_table(name).to_lance() for name in (*NODES, *EDGES)}, use_csr=True)
codebook = make_codebook()
D = len(codebook["ireland"])


def cypher(q):
    return engine.execute(q)

## A sample order history

These are the twelve past orders with an issue, read from the graph with one Cypher
query. Each one is an example of its class.

In [2]:
extract = '''
MATCH (o:Order)-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(c:Category),
      (o)-[:SHIPS_FROM]->(d:Depot),
      (o)-[:SHIPS_TO]->(r:Region)-[:IN_COUNTRY]->(k:Country),
      (o)-[:HAD_ISSUE]->(i:Issue)
RETURN o.name AS order_id, c.name AS category, d.name AS depot, r.name AS region, k.name AS country, i.name AS issue
ORDER BY issue, order_id
'''
history = pl.from_arrow(cypher(extract))
history

order_id,category,depot,region,country,issue
str,str,str,str,str,str
"""SO-0901""","""sofas""","""east-london""","""dublin""","""ireland""","""customs_delay"""
"""SO-0904""","""rugs""","""midlands""","""cork""","""ireland""","""customs_delay"""
"""SO-0908""","""desks""","""west-london""","""galway""","""ireland""","""customs_delay"""
"""SO-0912""","""lamps""","""midlands""","""ile-de-france""","""france""","""customs_delay"""
"""SO-0915""","""rugs""","""west-london""","""hauts-de-france""","""france""","""customs_delay"""
"""SO-0920""","""desks""","""east-london""","""cork""","""ireland""","""customs_delay"""
"""SO-0903""","""lamps""","""west-london""","""west-yorkshire""","""uk""","""damaged_in_transit"""
"""SO-0907""","""lamps""","""midlands""","""west-midlands""","""uk""","""damaged_in_transit"""
"""SO-0914""","""desks""","""west-london""","""merseyside""","""uk""","""damaged_in_transit"""


### Encode each order's motif

In notebook 3, we bundled the paths chosen for each query. Here we use the same four paths
for every order: product category, depot, destination region and country. `motif()` builds
those path vectors and bundles them, exactly as shown below.


In [3]:
def motif(codebook, category, depot, region, country):
    """Bundle the four paths used to compare orders in this classifier."""
    paths = [
        path(codebook, "CONTAINS", "IN_CATEGORY", category),
        path(codebook, "SHIPS_FROM", depot),
        path(codebook, "SHIPS_TO", region),
        path(codebook, "SHIPS_TO", "IN_COUNTRY", country),
    ]
    return torchhd.multiset(torch.stack(paths))


## One prototype per issue type

We want one hypervector per issue type that sums up what its cases look like.

This is done by bundling the motif hypervectors of the orders in each class:

$$\mathbf{P}_{\text{issue}} = \bigoplus_{o \,\in\, \text{issue}} \mathbf{M}(o)$$

We keep the bundle as a raw sum, without normalizing it back to ±1, so that adding one more
case later is just one more additive operation. We store the three prototypes in LanceDB, in a new table
called `issue_prototypes`.

In [4]:
def encode(row):
    return motif(codebook, row["category"], row["depot"], row["region"], row["country"])

before = {
    issue: torchhd.multiset(torch.stack([encode(row) for row in group.iter_rows(named=True)]))
    for (issue,), group in history.group_by("issue", maintain_order=True)
}
{issue: history.filter(pl.col("issue") == issue).height for issue in before}

{'customs_delay': 6, 'damaged_in_transit': 3, 'dispatch_hold': 3}

In [5]:
def store(prototypes):
    rows = [{"issue": issue, "vector": hv.tolist()} for issue, hv in prototypes.items()]
    return db.create_table("issue_prototypes", rows, mode="overwrite")

table = store(before)
table.count_rows()

3

### What the class prototype represents

A class prototype bundles its examples into a shared representation. Paths that occur
in several examples reinforce each other, giving common features more weight.

Here, we keep the raw sum. Taking its dot product with a path vector and dividing by $D$
approximates how many examples contributed that path. Other paths introduce small amounts
of interference because their vectors are nearly orthogonal.

The table below probes the features relevant to orders like SO-1042.


In [6]:
probes = {
    "lamps": path(codebook, "CONTAINS", "IN_CATEGORY", "lamps"),
    "east-london": path(codebook, "SHIPS_FROM", "east-london"),
    "ireland": path(codebook, "SHIPS_TO", "IN_COUNTRY", "ireland"),
    "sofas": path(codebook, "CONTAINS", "IN_CATEGORY", "sofas"),
    "midlands": path(codebook, "SHIPS_FROM", "midlands"),
}

def inspect_prototypes(prototypes):
    """Estimate how many examples in each class contributed each selected path."""
    return pl.DataFrame(
        [{"path": name} | {issue: round((proto @ probe).item() / D, 1) for issue, proto in prototypes.items()} for name, probe in probes.items()]
    )

inspect_prototypes(before)

path,customs_delay,damaged_in_transit,dispatch_hold
str,f64,f64,f64
"""lamps""",1.0,2.0,1.0
"""east-london""",2.1,0.1,1.0
"""ireland""",4.0,-0.1,1.0
"""sofas""",1.0,-0.1,2.0
"""midlands""",2.1,1.1,2.0


Read it column by column. The customs prototype is dominated by **Ireland** (4 of its 6
cases), and the damage prototype by **lamps** (2 of 3). The dispatch-hold prototype is mostly
the two **Midlands sofa** holds. The one lamps-from-East-London case, SO-0917, is outvoted
inside its own class.

## Classify the held-out orders

These six orders were resolved this week, so we know their true issue, but they're not in
the memory yet. That makes them a fair test.

To classify an order, we encode its motif and search `issue_prototypes` by cosine similarity. The most
similar prototype is the prediction.

In [7]:
pl.DataFrame(HELD_OUT)

order,category,depot,region,country,issue
str,str,str,str,str,str
"""SO-1051""","""lamps""","""east-london""","""limerick""","""ireland""","""dispatch_hold"""
"""SO-1053""","""lamps""","""east-london""","""greater-london""","""uk""","""dispatch_hold"""
"""SO-1055""","""desks""","""east-london""","""dublin""","""ireland""","""customs_delay"""
"""SO-1057""","""lamps""","""west-london""","""west-midlands""","""uk""","""damaged_in_transit"""
"""SO-1058""","""rugs""","""midlands""","""ile-de-france""","""france""","""customs_delay"""
"""SO-1060""","""sofas""","""midlands""","""greater-manchester""","""uk""","""dispatch_hold"""


We predict each test order's issue by finding the most similar class prototype.
The table compares these predictions with the known issues.


In [8]:
def classify(table):
    """Predict held-out orders' issues using the closest prototype and report scores and correctness."""
    rows = []
    for order in HELD_OUT:
        found = table.search(encode(order).numpy()).metric("cosine").limit(3).to_list()
        scores = dict(sorted((h["issue"], round(1 - h["_distance"], 3)) for h in found))  # fixed column order
        rows.append({"order": order["order"], "true": order["issue"], "predicted": found[0]["issue"]} | scores)
    return pl.DataFrame(rows).with_columns(correct=pl.col("true") == pl.col("predicted"))

results_before = classify(table)
results_before

order,true,predicted,customs_delay,damaged_in_transit,dispatch_hold,correct
str,str,str,f64,f64,f64,bool
"""SO-1051""","""dispatch_hold""","""customs_delay""",0.494,0.23,0.367,false
"""SO-1053""","""dispatch_hold""","""damaged_in_transit""",0.212,0.541,0.474,false
"""SO-1055""","""customs_delay""","""customs_delay""",0.634,0.102,0.243,true
"""SO-1057""","""damaged_in_transit""","""damaged_in_transit""",0.2,0.853,0.471,true
"""SO-1058""","""customs_delay""","""customs_delay""",0.509,0.113,0.241,true
"""SO-1060""","""dispatch_hold""","""dispatch_hold""",0.221,0.423,0.829,true


In [9]:
print("accuracy before:", results_before["correct"].sum(), "/", len(HELD_OUT))
assert results_before["correct"].sum() == 4

accuracy before: 4 / 6


Both missed orders are **East London lamp dispatch holds**, a pattern represented by only
one training case. SO-1051's Ireland destination favours the customs prototype, while
SO-1053's lamp and UK features favour the damage prototype. These overlaps plausibly
outweigh the underrepresented dispatch pattern.

## SO-1042 is confirmed as a dispatch hold

Say our sales colleague called the East London depot, as notebook 3 suggested. They learned that SO-1042 was held for the
same dangerous-goods paperwork as SO-0917. Two things get updated, one in each
representation.

**The graph** records that SO-1042 had a dispatch hold, along with the investigation note.
The table below retrieves the two matching dispatch-hold cases and their notes. Recording
this fact does not yet change the classifier.


In [10]:
# SO-1042 -[HAD_ISSUE]-> dispatch_hold, by node id
new_edge = {"src": 1042, "dst": 702, "note": "Same dangerous-goods hold at East London depot as SO-0917; released after calling the depot."}
db.create_table("HAD_ISSUE", edge_rows("HAD_ISSUE") + [new_edge], mode="overwrite")
engine.checkout_latest()  # the engine is pinned to the version it opened; move it to the new one

q = '''
MATCH (o:Order)-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(:Category {name: 'lamps'}),
      (o)-[:SHIPS_FROM]->(:Depot {name: 'east-london'}),
      (o)-[:SHIPS_TO]->(:Region)-[:IN_COUNTRY]->(:Country {name: 'ireland'}),
      (o)-[h:HAD_ISSUE]->(i:Issue)
RETURN o.name AS src, i.name AS dst, h.note AS note
'''
pl.from_arrow(cypher(q))

src,dst,note
str,str,str
"""SO-0917""","""dispatch_hold""","""Cordless lamps (lithium batteries) held at East London depot for dangerous-goods paperwork; released after calling the depot."""
"""SO-1042""","""dispatch_hold""","""Same dangerous-goods hold at East London depot as SO-0917; released after calling the depot."""


### Add the confirmed case to the class prototype

We define our classifier's **memory** is the three issue prototypes we built earlier. Each is a sum of
order motif hypervectors. Now that we know SO-1042's issue, we can add its motif to the correct sum.

We encode its category, depot, region and country using the same `motif()` function as
before. The confirmed issue selects the **dispatch-hold prototype**; it is not part of the
motif itself. We add the motif to that prototype and save the updated prototypes in LanceDB:

$$\mathbf{P}_{\text{dispatch\_hold}} \leftarrow \mathbf{P}_{\text{dispatch\_hold}} \oplus \mathbf{M}(\text{SO-1042})$$

The other two prototypes stay unchanged. We then classify the test orders again to see
whether this additional example changes the predictions.

![A new example hypervector added to an existing memory, illustrated with vector strips and a plus sign](../img/hdc-learning.png)

*SO-1042 is one additional resolved case: add its motif to the existing dispatch-hold prototype. A similar case, SO-0917, was already in memory.*

In [11]:
# Encode SO-1042's category, depot, destination region and country as one motif.
so_1042 = motif(codebook, "lamps", "east-london", "dublin", "ireland")

# Keep the original prototypes so we can compare predictions before and after learning.
after = dict(before)
# Add the confirmed order's motif to the dispatch-hold prototype.
after["dispatch_hold"] = before["dispatch_hold"] + so_1042
# Save the updated prototypes in LanceDB for the next classification run.
table = store(after)

# Inspect how strongly each prototype represents the selected paths.
inspect_prototypes(after)


path,customs_delay,damaged_in_transit,dispatch_hold
str,f64,f64,f64
"""lamps""",1.0,2.0,2.0
"""east-london""",2.1,0.1,2.0
"""ireland""",4.0,-0.1,2.0
"""sofas""",1.0,-0.1,2.0
"""midlands""",2.1,1.1,2.1


In the dispatch-hold column, lamps and East London now count 2 each, as many as the Midlands
sofa holds. Now we re-classify the same six orders.

In [12]:
results_after = classify(table)
results_after

order,true,predicted,customs_delay,damaged_in_transit,dispatch_hold,correct
str,str,str,f64,f64,f64,bool
"""SO-1051""","""dispatch_hold""","""dispatch_hold""",0.494,0.23,0.573,true
"""SO-1053""","""dispatch_hold""","""dispatch_hold""",0.212,0.541,0.566,true
"""SO-1055""","""customs_delay""","""customs_delay""",0.634,0.102,0.474,true
"""SO-1057""","""damaged_in_transit""","""damaged_in_transit""",0.2,0.853,0.468,true
"""SO-1058""","""customs_delay""","""customs_delay""",0.509,0.113,0.201,true
"""SO-1060""","""dispatch_hold""","""dispatch_hold""",0.221,0.423,0.672,true


In [13]:
print("accuracy after:", results_after["correct"].sum(), "/", len(HELD_OUT))
assert results_after["correct"].all()

accuracy after: 6 / 6


The classifier now predicts the correct issue for all six test orders.

## Before and after

Let's calculate the similarity scores for the before → after. A score can rise or fall without
changing the prediction: what matters is which class prototype is closest.

In [14]:
comparison = pl.DataFrame({
    "order": results_before["order"],
    "true": results_before["true"],
    "before": results_before["predicted"],
    "after": results_after["predicted"],
    "dispatch_hold score": [f"{b} → {a}" for b, a in zip(results_before["dispatch_hold"], results_after["dispatch_hold"])],
})
comparison

order,true,before,after,dispatch_hold score
str,str,str,str,str
"""SO-1051""","""dispatch_hold""","""customs_delay""","""dispatch_hold""","""0.367 → 0.573"""
"""SO-1053""","""dispatch_hold""","""damaged_in_transit""","""dispatch_hold""","""0.474 → 0.566"""
"""SO-1055""","""customs_delay""","""customs_delay""","""customs_delay""","""0.243 → 0.474"""
"""SO-1057""","""damaged_in_transit""","""damaged_in_transit""","""damaged_in_transit""","""0.471 → 0.468"""
"""SO-1058""","""customs_delay""","""customs_delay""","""customs_delay""","""0.241 → 0.201"""
"""SO-1060""","""dispatch_hold""","""dispatch_hold""","""dispatch_hold""","""0.829 → 0.672"""


## Our classifier learned from one additional example

We added SO-1042 to the dispatch-hold prototype, and that changed how it compares with
other orders. The before-and-after scores above show where that single example made a difference:

- **SO-1051 and SO-1053 now match it more closely**, enough for dispatch hold to beat the other classes.
- **SO-1055's score also rises**, but customs remains its closest prototype. A higher score doesn't necessarily change the prediction.
- **SO-1060's score falls**, because the prototype now represents more East London lamps alongside Midlands sofas. Dispatch hold still wins.

Note that the update changes the balance of features in a shared prototype, so it can affect many
orders, not just the example we added.

> Adding one confirmed case to one prototype took the classifier from four correct
predictions to all six, without a training loop.


## Wrap-up

A prototype is the bundle of a class's hypervectors (in this case, paths in a graph). Classifying an order is one
similarity search over three rows, and learning from a new case is one addition. There's no
training loop, no gradient, and nothing else to re-run.

Each representation kept the part it's good at. The graph recorded **why**: an explicit
`HAD_ISSUE` edge with a note you can audit. The hypervector memory recorded **what it looks
like**: a pattern that now steers the next lamp order from East London towards the depot.

Across the four notebooks we covered this broad theme:

1. **Graph**: connections stored explicitly; Cypher finds exact patterns.
2. **Operators**: ⊗, ⊕ and ρ turn structure into fixed-size hypervectors.
3. **Associative search**: a pattern becomes one hypervector; associative search finds the same
   precedent as Cypher, and ranks the near misses.
4. **Sample-efficient learning**: prototypes summarise cases by type, and can learn associations from a single new example.

> **A graph models connections explicitly. Hypervectors model them implicitly. Both answer
> questions about connected data, and they're stronger together.**